# Buổi 6 — Tư duy phản biện với dữ liệu PISA: thi thích ứng, cụm, đa kiểm định, trọng số

In [1]:
# Chạy ô này đầu tiên. Dữ liệu (PISA 2025, Việt Nam) được tải trực tiếp từ GitHub ở ô kế tiếp — không cần tải/upload tay.
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf, statsmodels.api as sm
sns.set_theme(style="whitegrid"); plt.rcParams["figure.figsize"]=(7,4); pd.set_option("display.precision",3)

> 📥 **Đầu vào:** nạp thư viện.

In [2]:
# ---- TẢI DỮ LIỆU (2 bảng: cấp trường 195 dòng, cấp học sinh 7.368 dòng) ----
RAW = "https://raw.githubusercontent.com/TatcataiTTN/for-Social-Science/main/SPSS/data/sav/"
truong = pd.read_csv(RAW + "vnm_truong_195_tong_hop.csv")
hs = pd.read_csv(RAW + "vnm_hocsinh_7368.csv")
VUNG={1:"ĐB sông Cửu Long",2:"Bắc TB, DH TB & Tây Nguyên",3:"Trung du & MN phía Bắc",4:"ĐB sông Hồng",5:"Đông Nam Bộ"}
truong["vung_ten"]=truong.vung.map(VUNG); truong["loai"]=truong.PRIVATESCH.map({1:"Công",2:"Tư"})
W=["EDULEAD","NEGSCLIM","STAFFSHORT","EDUSHORT","DIGPREP","AVLRSOFT","ENCOURPG"]
print("Bảng trường:",truong.shape,"| Bảng học sinh:",hs.shape)

Bảng trường: (195, 36) | Bảng học sinh: (7368, 54)


> 📥 **Đầu vào:** tải lại 2 bảng PISA VN.
>
> 📤 **Đầu ra thật:** `(195, 36)` và `(7368, 54)`. ✅

## 1. Thi thích ứng làm điểm thô 'phẳng'

In [3]:
lin=hs[hs.SPATH==1].sci_mc.dropna(); ada=hs[hs.SPATH==2].sci_mc.dropna()
print(f"Tuyến tính: {lin.mean():.3f} (n={len(lin)}) | Thích ứng: {ada.mean():.3f} (n={len(ada)}) |",stats.ttest_ind(lin,ada,equal_var=False))

Tuyến tính: 0.456 (n=1548) | Thích ứng: 0.454 (n=4489) | TtestResult(statistic=np.float64(0.40947812920190474), pvalue=np.float64(0.682224859403098), df=np.float64(2433.009664022239))


> 📥 **Đầu vào:** so sánh điểm Khoa học thô (`sci_mc`) giữa 2 nhóm học sinh làm bài theo **đường TUYẾN TÍNH** (`SPATH==1`, mọi học sinh làm cùng bộ câu hỏi độ khó cố định, n=1.548) và đường **THÍCH ỨNG** (`SPATH==2`, PISA hiện đại điều chỉnh độ khó câu hỏi theo năng lực học sinh trong lúc làm bài — thi thích ứng máy tính, n=4.489, chiếm đa số).
>
> 📤 **Đầu ra thật:** Tuyến tính mean=**0,456**; Thích ứng mean=**0,454** — GẦN NHƯ GIỐNG HỆT NHAU, `t=0,409, p=0,682` (hoàn toàn không có ý nghĩa thống kê).
>
> 🎯 **Trả lời câu hỏi ❓ bên dưới:** đây CHÍNH XÁC là điều thiết kế thi thích ứng CỐ Ý tạo ra — với đường thích ứng, học sinh giỏi bị "giao" câu hỏi khó hơn (để đo chính xác năng lực thật của họ), khiến TỈ LỆ TRẢ LỜI ĐÚNG THÔ của họ hội tụ về mức tương tự học sinh trung bình (~45%), dù NĂNG LỰC THẬT (đo qua mô hình IRT, không phải điểm thô) của họ cao hơn nhiều. Nói cách khác: **điểm thô (% câu đúng) giữa 2 đường thi KHÔNG THỂ so sánh trực tiếp** — phải dùng điểm năng lực đã ước lượng qua mô hình IRT (như các biến WLE đã dùng xuyên suốt các notebook trước), KHÔNG dùng % câu trả lời đúng thô, nếu không sẽ đưa ra kết luận sai lầm rằng "2 nhóm thi kiểu khác nhau nhưng năng lực như nhau" trong khi thực ra chỉ là hiệu ứng của thiết kế bài thi.

**❓** Đường thích ứng giao câu khó cho em giỏi. Điều đó làm tỉ lệ đúng thô hội tụ về ~45%. Có thể dùng `sci_mc` để nói 'em nào giỏi hơn' không?

## 2. Thiết kế xoay vòng: không em nào làm đủ 3 lĩnh vực

In [4]:
h=hs; A=(h.sci_n>0); B=(h.math_n>0); C=(h.read_n>0)
print("Khoa học:",A.sum(),"| Toán:",B.sum(),"| Đọc:",C.sum(),"| KH+Toán:",(A&B).sum(),"| KH+Đọc:",(A&C).sum(),"| Toán+Đọc:",(B&C).sum(),"| cả 3:",(A&B&C).sum())
print(h[["sci_mc","math_mc","read_mc","ldw_mc"]].corr().round(2))

Khoa học: 6037 | Toán: 3133 | Đọc: 3161 | KH+Toán: 2258 | KH+Đọc: 2280 | Toán+Đọc: 430 | cả 3: 0
         sci_mc  math_mc  read_mc  ldw_mc
sci_mc     1.00     0.61     0.51    0.56
math_mc    0.61     1.00     0.66    0.61
read_mc    0.51     0.66     1.00    0.49
ldw_mc     0.56     0.61     0.49    1.00


> 📥 **Đầu vào:** đếm số học sinh có dữ liệu ở từng lĩnh vực (Khoa học/Toán/Đọc) và các tổ hợp — vì PISA dùng **thiết kế ma trận xoay vòng** (rotated booklet design): mỗi học sinh chỉ làm MỘT PHẦN các lĩnh vực, không ai làm đủ cả 3, để giảm thời gian thi cho từng em nhưng vẫn phủ được toàn bộ nội dung ở cấp tổng thể.
>
> 📤 **Đầu ra thật:** Khoa học: 6.037 em; Toán: 3.133 em; Đọc: 3.161 em; KH+Toán: 2.258; KH+Đọc: 2.280; Toán+Đọc: chỉ 430 (rất ít); **cả 3: 0** — ĐÚNG NHƯ TIÊU ĐỀ mục 2, không một học sinh nào làm đủ cả 3 lĩnh vực! Ma trận tương quan giữa 4 biến năng lực (kể cả `ldw_mc`) dao động 0,49-0,66 — tương quan vừa-cao, hợp lý vì các năng lực học thuật thường liên quan nhau.
>
> ⚠️ **Hệ quả thực tế:** đây chính là lý do các hồi quy ở notebook buổi 4 (`sci_mc~math_mc`, `sci_mc~read_mc`...) mỗi lần có cỡ mẫu KHÁC NHAU (2.258, 2.280, 1.479) — không phải do dữ liệu thiếu ngẫu nhiên, mà do THIẾT KẾ THI CỐ Ý không cho học sinh làm đủ mọi lĩnh vực. Đây cũng là lý do vì sao phân tích PISA cấp học sinh phức tạp hơn nhiều so với dữ liệu lớp học đơn giản (240 học sinh, ai cũng có đủ mọi điểm số) đã dùng ở các module chính của site.

## 3. Cụm: 6.037 học sinh nhưng cỡ mẫu hiệu dụng chỉ ~640 (Buổi 3, phần 4)

## 4. Đa kiểm định: 7 chỉ số × 3 lĩnh vực = 21 tương quan

In [5]:
res=[(c,o,*stats.pearsonr(truong[c],truong[o])) for c in W for o in ["sci_mean","math_mean","read_mean"]]
r=pd.DataFrame(res,columns=["chỉ số","kết quả","r","p"]); print("p<0.05:",int((r.p<.05).sum()),"/21 | p<0.05/21 (Bonferroni):",int((r.p<.05/21).sum())); r[r.p<.05].round(3)

p<0.05: 2 /21 | p<0.05/21 (Bonferroni): 0


,chỉ số,kết quả,r,p
16,AVLRSOFT,math_mean,0.149,0.038
17,AVLRSOFT,read_mean,0.143,0.047


> 📥 **Đầu vào:** chạy TOÀN BỘ 21 kiểm định tương quan Pearson cùng lúc (7 chỉ số WLE × 3 kết quả `sci_mean/math_mean/read_mean`) — minh hoạ trực tiếp vấn đề "đa kiểm định" (multiple comparisons) đã cảnh báo ở Module 06/07 của site.
>
> 📤 **Đầu ra thật:** `p<0,05: 2/21` (chỉ AVLRSOFT với math_mean r=0,149 p=0,038 và với read_mean r=0,143 p=0,047 — CẢ HAI ĐỀU RẤT SÁT NGƯỠNG 0,05!) nhưng sau khi áp dụng hiệu chỉnh Bonferroni (`p<0,05/21≈0,00238`): **`p<0,05/21: 0/21`** — KHÔNG CÒN kiểm định nào có ý nghĩa!
>
> 🎯 **Trả lời câu hỏi ❓ bên dưới — xác suất có ≥1 dương tính giả:** nếu chạy 21 kiểm định độc lập mà KHÔNG có liên hệ thật nào (H0 đúng ở cả 21), xác suất có ÍT NHẤT 1 kiểm định cho p&lt;0,05 chỉ do ngẫu nhiên là `1-(1-0,05)^21 ≈ 1-0,95^21 ≈ **66%**` — tức 2/3! Kết quả 2/21 quan sát được ở đây HOÀN TOÀN PHÙ HỢP với những gì ngẫu nhiên thuần tuý có thể tạo ra, KHÔNG đủ bằng chứng để khẳng định `AVLRSOFT` (khả năng dùng phần mềm học tập) thực sự liên quan tới kết quả học tập.
>
> 🚨 **Đây là minh chứng số liệu THẬT, mạnh mẽ nhất trong toàn bộ 10 notebook của khoá học,** cho bài học cốt lõi về "p-hacking"/multiple comparisons: nếu một nhà nghiên cứu chỉ chạy 21 kiểm định rồi CHỌN RA đúng 2 kết quả "có ý nghĩa" (AVLRSOFT) để báo cáo mà bỏ qua 19 kết quả còn lại — mà không công bố rằng đã chạy tất cả 21 — đó chính là "đào bới p-value" (p-hacking), một trong những nguyên nhân chính gây ra khủng hoảng tái lặp (replication crisis) đã nhắc tới ở Module 04 của site.

**❓** Xác suất kỳ vọng có ≥1 p<.05 trong 21 kiểm định *nếu không có liên hệ thật* là bao nhiêu (1−0.95²¹)? Bạn có nên viết bài về 'phần mềm học tập liên quan điểm Toán' chỉ từ kết quả này?

## 5. Dự án cuối khoá
Chọn **một** câu hỏi nghiên cứu về trường học VN từ dữ liệu này, điền khung 7 mục (xem `bai_tap/RUBRIC.md`), viết 1 đoạn Kết quả thận trọng (nêu rõ: điểm thô, không có trọng số học sinh, có cụm, đa kiểm định).